## Example 1: S3 Service Mocking

In [ ]:
from moto import mock_aws
import boto3
import json

# Example 1: S3 bucket and object operations with mocking
with mock_aws():
    s3 = boto3.client('s3', region_name='us-east-1')
    
    # Create bucket and objects
    s3.create_bucket(Bucket='test-bucket')
    s3.put_object(Bucket='test-bucket', Key='data.json', 
                  Body=json.dumps({"users": [{"id": 1, "name": "Alice"}]}))
    s3.put_object(Bucket='test-bucket', Key='config.txt', Body=b'debug=true')
    
    # List buckets and objects
    buckets = s3.list_buckets()
    print(f"Buckets: {[b['Name'] for b in buckets['Buckets']]}")
    
    objects = s3.list_objects_v2(Bucket='test-bucket')
    print(f"Objects: {[obj['Key'] for obj in objects['Contents']]}")
    
    # Retrieve and process data
    response = s3.get_object(Bucket='test-bucket', Key='data.json')
    data = json.loads(response['Body'].read().decode())
    print(f"User data: {data['users'][0]['name']}")
    
    # Test bucket policies and configurations
    s3.put_bucket_versioning(Bucket='test-bucket', 
                           VersioningConfiguration={'Status': 'Enabled'})
    versioning = s3.get_bucket_versioning(Bucket='test-bucket')
    print(f"Versioning enabled: {versioning['Status']}")
    assert [b['Name'] for b in buckets['Buckets']] == ['test-bucket']
    assert sorted(obj['Key'] for obj in objects['Contents']) == ['config.txt', 'data.json']
    assert data == {"users": [{"id": 1, "name": "Alice"}]}
    assert versioning['Status'] == 'Enabled'

print("S3 mocking completed successfully")

## Example 2: EC2 Instance Management

In [ ]:
from moto import mock_aws
import boto3

# Example 2: EC2 instance lifecycle and management
with mock_aws():
    ec2 = boto3.client('ec2', region_name='us-west-2')
    
    # Create security group
    sg_response = ec2.create_security_group(
        GroupName='test-sg', Description='Test security group')
    sg_id = sg_response['GroupId']
    print(f"Created security group: {sg_id}")
    
    # Launch instances
    response = ec2.run_instances(
        ImageId='ami-12345678', MinCount=2, MaxCount=2,
        InstanceType='t2.micro', SecurityGroupIds=[sg_id])
    
    instance_ids = [inst['InstanceId'] for inst in response['Instances']]
    print(f"Launched instances: {instance_ids}")
    
    # Tag instances
    ec2.create_tags(Resources=instance_ids, 
                   Tags=[{'Key': 'Environment', 'Value': 'Test'}])
    
    # Describe instances and their states
    instances = ec2.describe_instances()
    for reservation in instances['Reservations']:
        for instance in reservation['Instances']:
            tags = {tag['Key']: tag['Value'] 
                   for tag in instance.get('Tags', [])}
            print(f"Instance {instance['InstanceId']}: "
                 f"{instance['State']['Name']}, Tags: {tags}")
    
    assert sg_id.startswith('sg-') and len(instance_ids) == 2
    described = {inst['InstanceId']: inst for res in instances['Reservations'] for inst in res['Instances']}
    assert set(described) == set(instance_ids)
    assert all({t['Key']: t['Value'] for t in described[i]['Tags']} == {'Environment': 'Test'} for i in instance_ids)

    # Stop and terminate instances
    ec2.stop_instances(InstanceIds=[instance_ids[0]])
    ec2.terminate_instances(InstanceIds=[instance_ids[1]])
    states = {inst['InstanceId']: inst['State']['Name']
              for res in ec2.describe_instances(InstanceIds=instance_ids)['Reservations'] for inst in res['Instances']}
    print(f"States after stop/terminate: {states}")
    assert states[instance_ids[0]] in ('stopping', 'stopped'), states
    assert states[instance_ids[1]] in ('shutting-down', 'terminated'), states
    print("Instance operations completed")

print("EC2 mocking completed successfully")

## Example 3: DynamoDB Database Operations

In [ ]:
from moto import mock_aws
import boto3

# Example 3: DynamoDB table creation and data operations
with mock_aws():
    dynamodb = boto3.client('dynamodb', region_name='us-east-1')
    
    # Create table
    table_name = 'Users'
    dynamodb.create_table(
        TableName=table_name,
        KeySchema=[{'AttributeName': 'userId', 'KeyType': 'HASH'}],
        AttributeDefinitions=[{'AttributeName': 'userId', 'AttributeType': 'S'}],
        BillingMode='PAY_PER_REQUEST'
    )
    print(f"Created DynamoDB table: {table_name}")
    
    # Put items
    users = [
        {'userId': {'S': 'user1'}, 'name': {'S': 'Alice'}, 'age': {'N': '30'}},
        {'userId': {'S': 'user2'}, 'name': {'S': 'Bob'}, 'age': {'N': '25'}},
        {'userId': {'S': 'user3'}, 'name': {'S': 'Charlie'}, 'age': {'N': '35'}}
    ]
    
    for user in users:
        dynamodb.put_item(TableName=table_name, Item=user)
    print(f"Inserted {len(users)} users")
    
    # Query and scan operations
    response = dynamodb.get_item(
        TableName=table_name, Key={'userId': {'S': 'user1'}})
    if 'Item' in response:
        item = response['Item']
        print(f"Retrieved: {item['name']['S']}, age {item['age']['N']}")
    
    # Scan all items
    scan_response = dynamodb.scan(TableName=table_name)
    print(f"Total items in table: {scan_response['Count']}")
    assert item['name']['S'] == 'Alice' and item['age']['N'] == '30'
    assert scan_response['Count'] == 3
    
    # Update item
    dynamodb.update_item(
        TableName=table_name,
        Key={'userId': {'S': 'user2'}},
        UpdateExpression='SET #age = :age',
        ExpressionAttributeNames={'#age': 'age'},
        ExpressionAttributeValues={':age': {'N': '26'}}
    )
    print("Updated user2's age")
    updated = dynamodb.get_item(TableName=table_name, Key={'userId': {'S': 'user2'}})['Item']
    assert updated['age']['N'] == '26' and updated['name']['S'] == 'Bob'

print("DynamoDB mocking completed successfully")

## Example 4: Lambda Functions and IAM

In [ ]:
from moto import mock_aws
import boto3
import zipfile
import io

# Example 4: Lambda functions with IAM roles
with mock_aws():
    # Create IAM role for Lambda
    iam = boto3.client('iam', region_name='us-east-1')
    lambda_client = boto3.client('lambda', region_name='us-east-1')
    
    assume_role_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }]
    }
    
    role_response = iam.create_role(
        RoleName='lambda-execution-role',
        AssumeRolePolicyDocument=str(assume_role_policy))
    role_arn = role_response['Role']['Arn']
    print(f"Created IAM role: {role_arn}")
    
    # Create Lambda function
    function_code = '''
def lambda_handler(event, context):
    name = event.get('name', 'World')
    return {'message': f'Hello, {name}!'}
'''
    
    # Create zip file in memory
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, 'w') as zip_file:
        zip_file.writestr('lambda_function.py', function_code)
    zip_buffer.seek(0)
    
    response = lambda_client.create_function(
        FunctionName='test-function', Runtime='python3.9',
        Role=role_arn, Handler='lambda_function.lambda_handler',
        Code={'ZipFile': zip_buffer.read()})
    
    print(f"Created Lambda function: {response['FunctionName']}")
    
    # List functions
    functions = lambda_client.list_functions()
    print(f"Total Lambda functions: {len(functions['Functions'])}")
    assert role_arn.endswith(':role/lambda-execution-role'), role_arn
    assert response['FunctionName'] == 'test-function'
    assert [f['FunctionName'] for f in functions['Functions']] == ['test-function']

print("Lambda and IAM mocking completed successfully")

### Invoking the function (needs Docker)

moto runs a Lambda handler inside a Docker container. That needs the `docker` Python package, which a plain
`pip install moto` does not install (`pip install "moto[awslambda]"` does), a reachable Docker daemon, and on
first use the daemon pulls `busybox` and a Lambda runtime image. The next cell is therefore tagged
`requires-docker` and `requires-docker-registry`; it creates the function again because each `mock_aws()`
block starts from an empty account.

In [ ]:
from moto import mock_aws
import boto3
import io
import json
import zipfile

# Example 4 (continued): invoke the Lambda function; moto runs the handler in a container
with mock_aws():
    iam = boto3.client('iam', region_name='us-east-1')
    lambda_client = boto3.client('lambda', region_name='us-east-1')
    role_arn = iam.create_role(
        RoleName='lambda-execution-role',
        AssumeRolePolicyDocument=json.dumps({
            "Version": "2012-10-17",
            "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"},
                           "Action": "sts:AssumeRole"}]}))['Role']['Arn']

    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, 'w') as zip_file:
        zip_file.writestr('lambda_function.py',
                          "def lambda_handler(event, context):\n"
                          "    name = event.get('name', 'World')\n"
                          "    return {'message': f'Hello, {name}!'}\n")
    lambda_client.create_function(
        FunctionName='test-function', Runtime='python3.9',
        Role=role_arn, Handler='lambda_function.lambda_handler',
        Code={'ZipFile': zip_buffer.getvalue()})

    invoke_response = lambda_client.invoke(
        FunctionName='test-function',
        Payload='{"name": "Moto"}')
    result = invoke_response['Payload'].read().decode()
    print(f"Lambda response: {result}")
    assert invoke_response['StatusCode'] == 200 and 'FunctionError' not in invoke_response, invoke_response
    assert json.loads(result) == {'message': 'Hello, Moto!'}, result

print("Lambda invocation completed successfully")

## Example 5: Multi-Service Integration Testing

In [ ]:
from moto import mock_aws
import boto3
import json
import time

# Example 5: Complex multi-service AWS application testing
with mock_aws():
    # Initialize multiple AWS services
    s3 = boto3.client('s3', region_name='us-east-1')
    dynamodb = boto3.resource('dynamodb', region_name='us-east-1')
    sns = boto3.client('sns', region_name='us-east-1')
    sqs = boto3.client('sqs', region_name='us-east-1')
    
    # Set up S3 data lake
    s3.create_bucket(Bucket='data-lake')
    s3.put_object(Bucket='data-lake', Key='events/2024/event1.json', 
                  Body=json.dumps({"user_id": "u1", "action": "login"}))
    print("Created S3 data lake with event data")
    
    # Create DynamoDB user table
    table = dynamodb.create_table(
        TableName='UserProfiles',
        KeySchema=[{'AttributeName': 'user_id', 'KeyType': 'HASH'}],
        AttributeDefinitions=[{'AttributeName': 'user_id', 'AttributeType': 'S'}],
        BillingMode='PAY_PER_REQUEST'
    )
    
    # Add user profiles
    table.put_item(Item={'user_id': 'u1', 'name': 'Alice', 'notifications': True})
    print("Created user profiles in DynamoDB")
    
    # Set up SNS topic for notifications
    topic_response = sns.create_topic(Name='user-notifications')
    topic_arn = topic_response['TopicArn']
    
    # Create SQS queue and subscribe to SNS
    queue_response = sqs.create_queue(QueueName='notification-queue')
    queue_url = queue_response['QueueUrl']
    queue_attrs = sqs.get_queue_attributes(QueueUrl=queue_url, AttributeNames=['QueueArn'])
    queue_arn = queue_attrs['Attributes']['QueueArn']
    
    sns.subscribe(TopicArn=topic_arn, Protocol='sqs', Endpoint=queue_arn)
    print("Set up SNS-SQS messaging pipeline")
    
    # Simulate event processing workflow
    event_data = s3.get_object(Bucket='data-lake', Key='events/2024/event1.json')
    event = json.loads(event_data['Body'].read().decode())
    
    # Check user preferences
    user = table.get_item(Key={'user_id': event['user_id']})['Item']
    if user.get('notifications'):
        # Send notification
        sns.publish(TopicArn=topic_arn, 
                   Message=f"User {user['name']} performed {event['action']}")
        
        # Check messages in queue
        messages = sqs.receive_message(QueueUrl=queue_url, MaxNumberOfMessages=1)
        if 'Messages' in messages:
            print(f"Notification delivered: {len(messages['Messages'])} message(s)")
        # SNS delivers to SQS wrapped in a JSON envelope whose "Message" field is the published text
        assert len(messages.get('Messages', [])) == 1, messages
        assert json.loads(messages['Messages'][0]['Body'])['Message'] == "User Alice performed login"
    assert event == {"user_id": "u1", "action": "login"}
    assert user['name'] == 'Alice' and user['notifications'] is True
    
    print("Multi-service integration test completed successfully")

print("Complex AWS workflow mocking completed successfully")